# VRL-YOLO-GUI — Train (Detection) on Colab

Companion notebook for the **Train on Colab** flow in VRL-YOLO-GUI. Run all cells, then copy the printed `trycloudflare.com` URL into the desktop app's **Connect to Colab** modal.

**Before you start:**
1. `Runtime → Change runtime type → GPU` (free Colab T4 is fine).
2. Upload your dataset to `MyDrive/VRL-YOLO-GUI/datasets/<dataset_name>/` with a Roboflow-style `data.yaml`.
3. Edit the `CONFIG` cell below to match your dataset name and training settings.
4. `Runtime → Run all`.

**Keep this cell session running** — when it stops, the tunnel URL stops working.

Documentation: see `docs/COLAB-GUIDE.md` in the repo.

In [ ]:
# 1. Mount Google Drive (your dataset lives here).
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Clone the VRL-YOLO-GUI repo (provides the Colab runtime modules)
#    and install Ultralytics. Re-running this cell pulls the latest fix.
import os, sys, subprocess
from pathlib import Path

REPO_DIR = Path('/content/vrl-yolo-gui-colab')
REPO_URL = 'https://github.com/atultiwari/VRL-YOLO-GUI.git'
BRANCH = 'main'

if REPO_DIR.is_dir():
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(REPO_DIR)], check=True)

# Add `notebooks/` so `_runtime.colab_*` imports resolve, and `server/`
# so `vrl_yolo.engine._runner_common` resolves (shared wire protocol).
for p in (REPO_DIR / 'notebooks', REPO_DIR / 'server'):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

# Ultralytics + uvicorn + fastapi are the only runtime deps we need.
# Colab already has fastapi; install the rest if missing.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics', 'uvicorn', 'fastapi'], check=True)

In [ ]:
# 3. CONFIG — edit these to match your dataset and training settings.
CONFIG = {
    'dataset_name': 'my-detect-dataset',   # folder under MyDrive/VRL-YOLO-GUI/datasets/
    'model':        'yolo26n.pt',           # yolo26{n,s,m,l,x}.pt or yolov8{n,s,m,l,x}.pt
    'epochs':       50,
    'imgsz':        640,
    'batch':        16,
}

# Resolved paths — generally no need to edit below this line.
DATASET_PATH = f'/content/drive/MyDrive/VRL-YOLO-GUI/datasets/{CONFIG["dataset_name"]}'
PROJECT_DIR  = '/content/vrl-yolo-gui-runs'   # Ultralytics output (Colab-local; served via tunnel)
RUN_NAME     = f'detect-{CONFIG["dataset_name"]}'
DEVICE       = 0   # first GPU; set to 'cpu' if you forgot to switch the runtime

In [ ]:
# 4. Start the server, open the Cloudflare tunnel, print the URL the
#    desktop needs. Token-authenticates every endpoint per docs/PLAN-P6.md §4.6.
import secrets
from pathlib import Path
from _runtime.colab_server import ColabServer
from _runtime.colab_tunnel import ensure_cloudflared, start_tunnel

TOKEN = secrets.token_urlsafe(16)
PORT  = 8765

server = ColabServer(
    token=TOKEN,
    task='detect',
    model=CONFIG['model'],
    epochs=int(CONFIG['epochs']),
)
server.start_uvicorn(port=PORT)

cloudflared = ensure_cloudflared(install_dir=Path('/content/.cloudflared'))
tunnel = start_tunnel(cloudflared=cloudflared, local_port=PORT)

PUBLIC_URL = f'{tunnel.url}?token={TOKEN}'
print('=' * 72)
print('Copy this URL into the desktop app\'s "Connect to Colab" modal:')
print()
print(f'    {PUBLIC_URL}')
print()
print('Keep this notebook running. The URL stops working when the cell stops.')
print('=' * 72)

In [ ]:
# 5. Run training. This blocks until training completes or you click Cancel
#    in the desktop UI. Live metric events stream through the tunnel.
from _runtime.colab_runner import run_training

exit_code = run_training(server, {
    'dataset': DATASET_PATH,
    'model':   CONFIG['model'],
    'task':    'detect',
    'project': PROJECT_DIR,
    'name':    RUN_NAME,
    'epochs':  int(CONFIG['epochs']),
    'imgsz':   int(CONFIG['imgsz']),
    'batch':   int(CONFIG['batch']),
    'device':  DEVICE,
})
print(f'\nTraining finished — exit code {exit_code}.')
print('Click "Save to library" in the desktop app to download best.pt.')
print('You can leave this cell idle until the desktop confirms the download.')